# KNN Phase-Level Predictor

Train K-Nearest Neighbors classifiers to predict DVFS phase labels from LIKWID-derived features.

In [1]:
# !pip install scikit-learn joblib

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

import joblib


In [3]:
# Datasets to train
datasets = [
    {
        "name": "AMDzen4c_edp",
        "csv_path": "../scripts/AMDzen4c/csv/training_dataset_dvfs_edp_labels_update.csv",
        "model_out": "artifacts/knn_amdzen4c_edp.joblib",
    },
    {
        "name": "AMDzen4c_energy",
        "csv_path": "../scripts/AMDzen4c/csv/training_dataset_dvfs_energy_labels_update.csv",
        "model_out": "artifacts/knn_amdzen4c_energy.joblib",
    },
]

features = [
    "CPI",
    "Compute_Density",
    "Mem_Boundness",
    "Stall_Ratio",
    "Branch_MPKI",
    "Vector_Intensity",
]

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

trained_models = {}

for ds in datasets:
    print(f"\n=== Training {ds['name']} ===")

    df = pd.read_csv(ds["csv_path"])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=features + ["label"], inplace=True)

    X = df[features]
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier()),
    ])

    # Tune KNN hyperparameters with CV
    param_grid = {
        "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["euclidean", "manhattan"],
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1_weighted",
        cv=5,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    print("Best Params:", grid.best_params_)
    print("=== Classification Report ===")
    print(classification_report(y_test, y_pred))
    print("=== Confusion Matrix ===")
    print(confusion_matrix(y_test, y_pred))

    model_path = Path(ds["model_out"])
    joblib.dump({
        "model": best_model,
        "features": features,
        "classes": list(best_model.named_steps["knn"].classes_),
        "best_params": grid.best_params_,
        "dataset": ds["name"],
    }, model_path)
    print(f"Saved model to {model_path}")

    trained_models[ds["name"]] = {
        "model": best_model,
        "features": features,
        "classes": list(best_model.named_steps["knn"].classes_),
        "best_params": grid.best_params_,
        "model_out": str(model_path),
    }



=== Training AMDzen4c_edp ===
Best Params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
=== Classification Report ===
              precision    recall  f1-score   support

    HighFreq       1.00      1.00      1.00      2845
     LowFreq       1.00      1.00      1.00      1298

    accuracy                           1.00      4143
   macro avg       1.00      1.00      1.00      4143
weighted avg       1.00      1.00      1.00      4143

=== Confusion Matrix ===
[[2844    1]
 [   0 1298]]
Saved model to artifacts/knn_amdzen4c_edp.joblib

=== Training AMDzen4c_energy ===
Best Params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
=== Classification Report ===
              precision    recall  f1-score   support

    HighFreq       1.00      1.00      1.00      2079
     LowFreq       1.00      1.00      1.00      1869
     MedFreq       1.00      1.00      1.00       195

    accuracy                           1.00 

In [4]:
# Example: load one saved KNN model and run a single prediction
example = joblib.load("artifacts/knn_amdzen4c_edp.joblib")
knn_model = example["model"]
feature_order = example["features"]

print("Loaded classes:", example["classes"])
print("Feature order:", feature_order)

# Replace with a real feature row from runtime profiling
sample = pd.DataFrame([{
    "CPI": 1.0,
    "Compute_Density": 1.0,
    "Mem_Boundness": 1.0,
    "Stall_Ratio": 1.0,
    "Branch_MPKI": 1.0,
    "Vector_Intensity": 1.0,
}])[feature_order]

pred = knn_model.predict(sample)[0]
print("Predicted phase level:", pred)


Loaded classes: ['HighFreq', 'LowFreq']
Feature order: ['CPI', 'Compute_Density', 'Mem_Boundness', 'Stall_Ratio', 'Branch_MPKI', 'Vector_Intensity']
Predicted phase level: HighFreq
